# Global registration with RANSAC
We are going to use [open3d](http://www.open3d.org/) to handle point clouds and generation of point clouds
We are importing the packages and defining a function which helps us drawing the point clouds.

In [1]:
import open3d as o3d
import numpy as np
import copy

# helper function for drawing
# If you want it to be more clear set recolor=True
def draw_registrations(source, target, transformation = None, recolor = True):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if(transformation is not None):
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

We need to read in our pointclouds. For that we use the `io` module of the
open3d package (`o3d`). The following cell will open a window with a
visualization of both point clouds `source` and `target`.

Also, this page [Visualization - Open3D](http://open3d.org/html/tutorial/Basic/visualization.html)
contains some useful examples and instructions on how to use the viewer.

In [5]:
source = o3d.io.read_point_cloud("ICP/r1.pcd")
target = o3d.io.read_point_cloud("ICP/r2.pcd")

# Used for downsampling.
voxel_size = 0.05

# Show models side by side
draw_registrations(source, target)

### Finding features in pointclouds
When working on point clouds it can be beneficial to work on a downsampled version of the point cloud,
as it decreases the need of computation.

You can use [`pointcloud.voxel_down_sample()`](http://www.open3d.org/docs/latest/python_api/open3d.geometry.PointCloud.html#open3d.geometry.PointCloud.voxel_down_sample) where `pointcloud` is the name of your point cloud object. In our case, that would be `source` and `target`.

We also need to estimate the normals of the point cloud points using [`pointcloud.estimate_normals()`](http://www.open3d.org/docs/latest/python_api/open3d.geometry.PointCloud.html#open3d.geometry.PointCloud.voxel_down_sample)

**Task:** Find FPFH features or correspondances of the downsampled point clouds.
[`o3d.pipelines.registration.compute_fpfh_feature()`](http://www.open3d.org/docs/latest/python_api/open3d.pipelines.registration.compute_fpfh_feature.html)


In [6]:
####
# Downsample and find features here
####

bbox = source.get_axis_aligned_bounding_box()
extent = bbox.get_extent()   # [x_length, y_length, z_length]
# print(extent)

# Commonly use voxel size is usually 1-5% of largest extent which is 3.41 
# in our case --> voxel size = 3.41*0.05 = 0.17

source_down = source.voxel_down_sample(0.17)
target_down = target.voxel_down_sample(0.17) 

# Estimate normals for both downsampled point clouds
radius_normal = 0.17 * 2  # Typically 2x the voxel size

source_down.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
)
target_down.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
)

# Compute FPFH features
radius_feature = 0.17 * 5  # Typically 5x the voxel size for features

source_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    source_down,
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100)
)
target_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    target_down,
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100)
)


### RANSAC 
We will now attempt to use RANSAC to do a global registration of the two point clouds.

By using the function [`o3d.pipelines.registration.registration_ransac_based_on_feature_matching`](http://www.open3d.org/docs/latest/python_api/open3d.pipelines.registration.registration_ransac_based_on_feature_matching.html) from open3d, do the following:


Try to find the transformation from `r1.pcd` (`source`) to `r2.pcd` (`target`).
Attempt with point-to-point and point-to-plane
```Python
point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()
```

When using RANSAC, focus on the arguments below. The rest are optional parameters.
```Python
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    distance_threshold,
    point_to_point)
```

In [7]:
####
# Call RANSAC here
####

distance_thresh = 0.2

point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()

ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down, target_down,
    source_fpfh, target_fpfh,
    True,  # mutual_filter
    distance_thresh,  # max_correspondence_distance
    point_to_point,  # estimation_method
)
# draw_registrations(source, target, ransac_result.transformation, True)

## Exercises
### A)
Can you get a decent transformation from r1 to r3? (check the ICP folder)
### B)
With the following checkers, can you get better results from RANSAC? Try tweaking the parameters of them. Can you make point-to-plane work? Do not spend too much time on this, if you can't manage, skip it. (I was not able to get a good fit.)

You can also try tweaking the `voxel_size`

```Python
corr_length = 0.9
distance_threshold = voxel_size * 1.5

c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0,c1,c2]

ransac_result = o3d.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    True,
    distance_threshold,
    point_to_point,
    checkers = checker_list)
```


In [8]:
r1 = o3d.io.read_point_cloud("ICP/r1.pcd")
r3 = o3d.io.read_point_cloud("ICP/r3.pcd")

# Used for downsampling.
voxel_size = 0.2

r1_d = source.voxel_down_sample(voxel_size)
r3_d = target.voxel_down_sample(voxel_size) 

# Estimate normals for both downsampled point clouds
radius_normal = voxel_size * 2  # Typically 2x the voxel size

r1_d.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
)
r3_d.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
)

# Compute FPFH features
radius_feature = voxel_size * 3  # Typically 5x the voxel size for features
r1_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    r1_d,
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100)
)
r3_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    r3_d,
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100)
)

####
# Call RANSAC here
####

distance_thresh = voxel_size*1.5

point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()

corr_length = 0.9
distance_threshold = voxel_size * 1.5


### Try to use checkers to get a better result
c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0,c1,c2]

ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    r1_d, r3_d,
    r1_fpfh, r3_fpfh,
    True,  # mutual_filter
    distance_thresh,  # max_correspondence_distance
    point_to_point,  # estimation_method,
    checkers=checker_list
)
draw_registrations(r1, r3, ransac_result.transformation, True)

In [9]:
####
# Local refinement with ICP
####

# If you want to use point-to-plane ICP, you MUST have normals on the *full* clouds:
r1.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
)
r3.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
)

# Use a smaller distance threshold for ICP than for RANSAC
icp_distance_thresh = voxel_size * 0.8

# Choose ICP variant:
#   - point_to_point: more robust when normals are noisy
#   - point_to_plane: converges faster / better when normals are good
icp_result = o3d.pipelines.registration.registration_icp(
    r1,                          # source (will be moved)
    r3,                          # target (fixed)
    icp_distance_thresh,         # max_correspondence_distance
    ransac_result.transformation,# initial guess from global alignment
    point_to_plane               # or point_to_point
)

print("RANSAC transformation:\n", ransac_result.transformation)
print("ICP refined transformation:\n", icp_result.transformation)

# Visualize ICP result
draw_registrations(r1, r3, icp_result.transformation, True)


RANSAC transformation:
 [[-0.61578891  0.78460719  0.07208029  0.78078976]
 [-0.08937421 -0.16044948  0.98298943  0.68299311]
 [ 0.78282582  0.59887187  0.16892665 -1.51260238]
 [ 0.          0.          0.          1.        ]]
ICP refined transformation:
 [[-0.27020819  0.9627985  -0.00256487 -0.0106593 ]
 [-0.25843967 -0.06996406  0.96349051  0.82692136]
 [ 0.92746777  0.26100589  0.26773019 -1.16061386]
 [ 0.          0.          0.          1.        ]]
